# Daily Merge

Merge raw session CSV files into one `merged_<animal>.csv` file per animal, then optionally merge all animals into `merged_all_subjects.csv` for the full cohort of the selected line.

## 1. Setup

Run this cell first. It makes imports work whether the notebook is launched from the repo root or from inside `notebooks/ASD` or `notebooks/Stakes`.

In [8]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "DailyMerge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/mafaldavalente/Documents/Mafalda_analysis')

## 2. Choose Dataset

Set `RAT = None` to process every animal in the cohort, or set it to one animal ID such as `"JCS0013"`.

In [25]:
LINE = "Stakes"
COHORT = "cohort2"
RAT = None  # e.g. "JCS0013", or None for all animals

MODE = "both"  # "session", "animals", or "both"
MODEL_FILE = None

## 3. Optional Session Removals

Edit `SESSION_EDITS` before merging if a daily CSV should be removed entirely, or if only the bad tail/range of a session should be removed or marked repeated. File names must match the raw CSV names inside each animal folder.


In [ ]:
# Optional per-animal cleanup rules applied while creating merged_<animal>.csv.
# These affect the merged output only; they do not edit the raw daily CSV files.
#
# Actions:
# - drop_entire_session: skip that raw CSV completely
# - drop_from_trial: remove trials with trial >= start_trial
# - drop_trial_range: remove trials from start_trial through end_trial, inclusive
# - drop_block: remove one or more block values from a session file
# - mark_repeated_from: keep rows but set repeated_trial = True from start_trial onward

SESSION_EDITS = {
    # Previous provisions from DailyMerge.py.
    "ASD0013": [
        {"file": "out_ASD0013_251014.csv", "action": "mark_repeated_from", "start_trial": 6690},
    ],

    # Previous ASD0018 provisions from DailyMerge.py.
    # Change action to "drop_from_trial" if you want these rows removed instead.
    "ASD0018": [
        {"file": "ASD0018_out_251014.csv", "action": "mark_repeated_from", "start_trial": 7370},
        {"file": "ASD0018_out_251015.csv", "action": "mark_repeated_from", "start_trial": 8000},
        {"file": "out_ASD0018_251028.csv", "action": "mark_repeated_from", "start_trial": 10900},
        {"file": "out_ASD0018_251127.csv", "action": "mark_repeated_from", "start_trial": 22250},
    ],

    "ASD0052": [
        {"file": "out_ASD0052_260707.csv", "action": "mark_repeated_from", "start_trial": 39640},
    ],

    "ASD0058": [
        {"file": "out_ASD0058_260617.csv", "action": "drop_block", "blocks": [1, 2, 3]},
    ],

    # Examples for ASD0019. Uncomment/edit the raw filenames and thresholds as needed.
    # "ASD0019": [
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_entire_session"},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_from_trial", "start_trial": 5000},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_trial_range", "start_trial": 1000, "end_trial": 1500},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "block": 2},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "blocks": [2, 3]},
    # ],
}

SESSION_EDITS


{'ASD0013': [{'file': 'out_ASD0013_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 6690}],
 'ASD0018': [{'file': 'ASD0018_out_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 7370},
  {'file': 'ASD0018_out_251015.csv',
   'action': 'mark_repeated_from',
   'start_trial': 8000},
  {'file': 'out_ASD0018_251028.csv',
   'action': 'mark_repeated_from',
   'start_trial': 10900},
  {'file': 'out_ASD0018_251127.csv',
   'action': 'mark_repeated_from',
   'start_trial': 22250}],
 'ASD0058': [{'file': 'out_ASD0058_260617.csv',
   'action': 'drop_block',
   'blocks': [1, 2, 3]}]}

## 4. Optional Bad RT Values

Use `RT_VALUE_EDITS` when task outcomes and abort labels are valid, but the recorded numeric `timed_rt` values should be ignored in RT analyses. These edits keep the trials and only set `timed_rt` to missing in the merged outputs.


In [11]:
# Numeric RT values to ignore while keeping trials for accuracy/choice/abort analyses.
# This only blanks timed_rt and adds rt_value_valid / rt_value_note columns.
# It does not change success, abort_type, choices, trial counts, or repeated_trial.

RT_VALUE_EDITS = [
    {
        "setup": 2,
        "start_date": "2026-06-13",
        "end_date": "2026-06-18",  # update if the setup-2 issue continues
        "date_col": "source_date",
        "setup_col": "box",
        "rt_col": "timed_rt",
        "reason": "setup 2 RT value recording issue",
    },

]

RT_VALUE_EDITS


[{'setup': 2,
  'start_date': '2026-06-13',
  'end_date': '2026-06-18',
  'date_col': 'source_date',
  'setup_col': 'box',
  'rt_col': 'timed_rt',
  'reason': 'setup 2 RT value recording issue'}]

## 5. Preview Animals

Check which animals will be processed before writing merged files.

In [22]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import get_animals_for_cohort, get_base_dir

base_dir = get_base_dir(LINE, COHORT)
animals = get_animals_for_cohort(LINE, COHORT, rat=RAT)

print(f"Base directory: {base_dir}")
print(f"Animals ({len(animals)}): {animals}")

Base directory: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/CNTNAP2_cohort3
Animals (16): ['ASD0022', 'ASD0025', 'ASD0026', 'ASD0027', 'ASD0028', 'ASD0029', 'ASD0030', 'ASD0031', 'ASD0032', 'ASD0033', 'ASD0034', 'ASD0035', 'ASD0036', 'ASD0037', 'ASD0050', 'ASD0051']


## 6. Merge Daily Files Per Animal

This creates or updates `merged_<animal>.csv` files in the cohort data folder.

In [23]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import merge_session_files

if MODE in ("session", "both"):
    merge_session_files(
        line=LINE,
        cohort=COHORT,
        rat=RAT,
        session_edits=SESSION_EDITS,
        rt_value_edits=RT_VALUE_EDITS,
    )
else:
    print("Skipping per-animal session merge.")

Processing 16 animal(s) for CNTNAP2 cohort3: ASD0022, ASD0025, ASD0026, ASD0027, ASD0028, ASD0029, ASD0030, ASD0031, ASD0032, ASD0033, ASD0034, ASD0035, ASD0036, ASD0037, ASD0050, ASD0051
Using latest file 'out_ASD0022_260615.csv' as column reference (75 columns).
Total unique columns across all files: 80
✅ Added out_ASD0022_251117.csv (685 rows)
✅ Added out_ASD0022_251118.csv (920 rows)
✅ Added out_ASD0022_251120.csv (661 rows)
✅ Added out_ASD0022_251124.csv (926 rows)
✅ Added out_ASD0022_251125.csv (922 rows)
✅ Added out_ASD0022_251126.csv (774 rows)
✅ Added out_ASD0022_251127.csv (759 rows)
✅ Added out_ASD0022_251128.csv (728 rows)
✅ Added out_ASD0022_251204.csv (710 rows)
✅ Added out_ASD0022_251205.csv (764 rows)
✅ Added out_ASD0022_251209.csv (686 rows)
✅ Added out_ASD0022_251211.csv (717 rows)
✅ Added out_ASD0022_251210.csv (750 rows)
✅ Added out_ASD0022_251215.csv (752 rows)
✅ Added out_ASD0022_251216.csv (731 rows)
✅ Added out_ASD0022_251217.csv (748 rows)
✅ Added out_ASD0022_2

## 7. Merge Animals Into Cohort File

This creates or updates `merged_all_subjects.csv` in the cohort data folder.

In [24]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import merge_subject_files_with_model

if MODE in ("animals", "both"):
    merged_df = merge_subject_files_with_model(
        line=LINE,
        cohort=COHORT,
        model_file=MODEL_FILE,
    )
else:
    merged_df = None
    print("Skipping cohort-level animal merge.")

if merged_df is not None:
    display(merged_df.head())
    print(merged_df.shape)

📘 Using model file 'merged_ASD0022.csv' with 86 columns.
🧾 Total unique columns across all subjects: 106


/Users/mafaldavalente/Documents/Mafalda_analysis/DailyMerge.py:623: DtypeWarning: Columns (0: opto_onset) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0029.csv (34430 rows)
✅ Added merged_ASD0028.csv (31565 rows)
✅ Added merged_ASD0051.csv (29753 rows)
✅ Added merged_ASD0050.csv (27376 rows)
✅ Added merged_ASD0037.csv (43224 rows)


/Users/mafaldavalente/Documents/Mafalda_analysis/DailyMerge.py:623: DtypeWarning: Columns (0: opto_mode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0022.csv (63793 rows)
✅ Added merged_ASD0036.csv (43512 rows)
✅ Added merged_ASD0034.csv (38517 rows)
✅ Added merged_ASD0035.csv (36522 rows)
✅ Added merged_ASD0025.csv (31228 rows)
✅ Added merged_ASD0031.csv (33560 rows)
✅ Added merged_ASD0030.csv (14280 rows)
✅ Added merged_ASD0032.csv (25429 rows)
✅ Added merged_ASD0026.csv (31621 rows)
✅ Added merged_ASD0027.csv (20481 rows)


/Users/mafaldavalente/Documents/Mafalda_analysis/DailyMerge.py:623: DtypeWarning: Columns (0: opto_onset, 1: rt_value_note) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0033.csv (25756 rows)
🎉 Saved merged dataset: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/CNTNAP2_cohort3/merged_all_subjects.csv
Final shape: (531047, 106)


,animal,batch,experimenter,version,bias,repeated_trial,trial,trial_start,tared_trial_start,trial_end,...,sound_ramp_time,base_ft,ft_exp_mean,catch_trial,lnp_end,intended_opto_onset_time,timed_opto_onset_time,opto_delay,trial_end_frame,lnp_end_frame
0,ASD0022,cntnap2,HY,0.6.1,0.00,True,1,3.846222e+09,0.000000,3.846222e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ASD0022,cntnap2,HY,0.6.1,0.00,True,2,3.846222e+09,182.238976,3.846222e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ASD0022,cntnap2,HY,0.6.1,0.00,True,3,3.846222e+09,364.269984,3.846223e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ASD0022,cntnap2,HY,0.6.1,-0.04,True,4,3.846223e+09,724.797984,3.846223e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ASD0022,cntnap2,HY,0.6.1,-0.08,True,5,3.846223e+09,878.237984,3.846223e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(531047, 106)
